In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
bronze_mapeamento= {
    'temp_bronze_clientes' : f'{bronze_path}/clientes/',
    'temp_bronze_itens_pedido' : f'{bronze_path}/itens_pedido/',
    'temp_bronze_pedidos' : f'{bronze_path}/pedidos/',
    'temp_bronze_produtos' : f'{bronze_path}/produtos/',
    'temp_bronze_vendedores' : f'{bronze_path}/vendedores/'

}
for view_name, path in bronze_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)

)

In [0]:
%sql
select * from temp_bronze_pedidos

In [0]:
%sql
describe temp_bronze_pedidos

In [0]:
%python
df_pedido = spark.sql("""
    SELECT DISTINCT
        id_pedido,
        id_cliente,
        id_vendedor,
        DATE_FORMAT(data_pedido, 'dd/MM/yyyy') as dataPedido,
        status_pedido,
        desconto

    FROM temp_bronze_pedidos

    WHERE LOWER(TRIM(status_pedido)) <> 'cancelado'
""")


# Salvar em delta na silver
df_pedido.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeSchema', 'true')\
                .save(f'{silver_path}/pedidos')

In [0]:
%python
display(df_pedido)